In [2]:
import transformers
from transformers.models.llama import modeling_llama
from transformers import LlamaConfig
import torch
from torch import nn, IntTensor, Tensor
LlamaDecoderLayer = modeling_llama.LlamaDecoderLayer # for some reason the import only works like this on my laptop
from tokenizer_layer import TokenizerLayer, TokenizerLayerV2
from vocab_manager import DynamicVocabHead

In [3]:
from typing import Optional
class TokenizerModel(nn.Module):
    def __init__(self,
                 config: LlamaConfig,
                 padding_side = "left",
                 num_tokenizer_layers = 1
                 ):
        super().__init__()
        hidden_size = config.hidden_size
        vocab = [(i,) for i in range(config.vocab_size)]
        self.embeder_layer = nn.Embedding(len(vocab), hidden_size)
        self.tokenizer_layers = [
        TokenizerLayerV2(config, i, vocab, passing_threshold=config.passing_threshold)
        for i in range(num_tokenizer_layers)
        ]
        self.lm_head = DynamicVocabHead(hidden_size, vocab)
        self.softmax = nn.Softmax(dim=-1)
    
    def forward(self,
                input_ids: IntTensor,
                attention_mask: Optional[Tensor] = None
                ):
        hidden_states = self.embeder_layer(input_ids)
        batched_tokens = []
        for x in input_ids.tolist():
            tupled_ids = list(map(lambda y: (y,), x))
            batched_tokens.append(tupled_ids)
        for tokenizer_layer in self.tokenizer_layers:
            hidden_states, new_batched_tokens, attention_mask =\
                tokenizer_layer(hidden_states, batched_tokens, attention_mask)
        head = self.lm_head.forward(hidden_states, new_batched_tokens)
        out = self.softmax(head)

        return out

In [4]:
pad_token_id = 256

config = LlamaConfig(
    vocab_size=257,
    hidden_size=512,
    intermediate_size=1024,
    num_hidden_layers=4,
    num_attention_heads=8,
    max_position_embeddings=512,
    rms_norm_eps=1e-6,
    initializer_range=0.02,
    use_cache=True,
    pad_token_id=pad_token_id,
    # bos_token_id=tokenizer.bos_token_id,
    # eos_token_id=tokenizer.eos_token_id,
    tie_word_embeddings=False,
    passing_threshold = 0.5
)

In [4]:
tokenizer_model = TokenizerModel(config, num_tokenizer_layers=2)

In [5]:
from datasets import load_dataset
utf_8_dataset = load_dataset(
    "json",
    data_files={"train": "train.json", "test": "test.json"},
    split="train"
    )
utf_8_dataset

Dataset({
    features: ['strings', 'input_ids', 'labels'],
    num_rows: 9000
})

In [6]:
import torch

def pad_sft_collator(batch, padding_value: int=257, padding_side = "left"):
    # TODO: implement this function
    tensor_batch_ids = [torch.tensor(x['input_ids']) for x in batch]
    tensor_batch_labels = [torch.tensor(x['labels']) for x in batch]

    input_ids = torch.nn.utils.rnn.pad_sequence(
        tensor_batch_ids,
        padding_value=padding_value,
        padding_side=padding_side,
        ).T
    labels = torch.nn.utils.rnn.pad_sequence(
        tensor_batch_labels,
        padding_value=padding_value,
        padding_side=padding_side,
        ).T

    attention_mask = []
    max_len = input_ids.shape[-1]
    for ids in tensor_batch_ids:
        ids_len = len(ids)
        if padding_side == "left":
            new_mask = torch.tensor([0]*(max_len-ids_len) + [1]*ids_len)
        else:
            new_mask = torch.tensor([1]*ids_len + [0]*(max_len-ids_len))
        attention_mask.append(new_mask)

    attention_mask = torch.stack(attention_mask)#torch.ones_like(input_ids)

    # of shape (batch size, sequence length), input_ids/labels is of dtype long, the mask bool
    return {
        'input_ids': input_ids,
        'labels': labels,
        'attention_mask': attention_mask,
    }

# test that the function runs fine
batch_size = 2
batch = utf_8_dataset.select(range(batch_size))

padded_batch = pad_sft_collator(batch, padding_value=256) # padding value should be beyond the range of numbers representable by one byte
padded_batch["input_ids"].shape

torch.Size([2, 198])

In [11]:
tokenizer_model(padded_batch["input_ids"]).shape

torch.Size([2, 97, 357])

In [30]:
list(tokenizer_model.lm_head.vocab_manger.vocab.iloc[288].token_id)

[240, 171, 191]